In [1]:
import numpy as np
import torch
import json
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.transforms import ToTensor
from torchvision.ops import generalized_box_iou_loss
from imageToHandBoxCNN import ImageToHandBoxCNN
from fullImageDataSet import FullImageDataSet

In [2]:
MODEL_PATH = "imageToHandBox_model.pth"
IMAGES_DIR = 'data/AUGMENTED_IMAGES'
ANNOTATIONS_DIR = 'data/AUGMENTED_IMAGES/augmented_annotations.json'

transform = transforms.Compose([
    transforms.ToTensor()
])

full_image_train_dataset = FullImageDataSet(image_dir=IMAGES_DIR,
                                      annotation_dir=ANNOTATIONS_DIR,
                                      transform=transform)

full_image_test_dataset = FullImageDataSet(image_dir=IMAGES_DIR,
                                      annotation_dir=ANNOTATIONS_DIR,
                                      transform=transform,
                                      train=False)

BATCH_SIZE = 128

trainDataLoader = DataLoader(full_image_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
testDataLoader = DataLoader(full_image_test_dataset, batch_size=BATCH_SIZE)
for X, y in trainDataLoader:
    print(f"Shape of X: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

device = ("cuda"
          if torch.cuda.is_available()
          #else "mps"
          #if torch.backends.mps.is_available()
          else "cpu"
         )
print(f"Using {device} as device")

Shape of X: torch.Size([128, 3, 512, 512])
Shape of y: torch.Size([128, 4]) torch.float32
Using cpu as device


In [3]:
model = ImageToHandBoxCNN().to(device)
print(model)

ImageToHandBoxCNN(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (convolutional_relu_stack): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU()
    (6): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU()
    (10): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (linear_stack): Sequential(
    (0): Linear(in_features=262144, out_features=1024, bias=True)


In [4]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def bbox_loss(pred, target):
    return F.smooth_l1_loss(pred, target, beta=1.0)

loss_fn = bbox_loss

In [5]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)

        pred = model(x)
        
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if (batch) % 10 == 0:
            loss, current = loss.item(), (batch + 1) * len(x)
            print(f"Loss: {loss:>7f} [{current:>5d}/{size:>5d}]")
            #print(f"Current pred: {pred.tolist()}")
            #print(f"Current real: {y.tolist()}")

In [6]:
def accuracy_fn(predicted, target):
    predicted = predicted.tolist()
    target = target.tolist()

    accuracies = []
    
    for index in range(len(target)):
        predicted_current = predicted[index]
        target_current = target[index]
        
        pXMin, pYMin, pXMax, pYMax = predicted_current
        tXMin, tYMin, tXMax, tYMax = target_current
    
        XMin = max(pXMin, tXMin)
        YMin = max(pYMin, tYMin)
        XMax = min(pXMax, tXMax)
        YMax = min(pYMax, tYMax)
    
        intersection = max(0, XMax - XMin) * max(0, YMax - YMin)
    
        pArea = (pXMax - pXMin) * (pYMax - pYMin)
        tArea = (tXMax - tXMin) * (tYMax - tYMin)
    
        union = pArea + tArea - intersection
    
        iou = intersection / union if union > 0 else 0

        accuracies.append(iou)
    return sum(accuracies) / len(accuracies)

def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            test_loss += loss_fn(pred, y).item()
            correct += accuracy_fn(pred, y)

    average_loss = test_loss / num_batches
    accuracy = correct / size
    print(f"Accuracy: {accuracy}")
    print(f"Average Loss: {average_loss}")
    return accuracy, average_loss

In [7]:
def train_epochs(epochs, given_model, loss_fn, optimizer, forgiveness=1, minimum_epochs=5):
    losses = []
    accuracies = []
    average_loss_min = float('inf')
    misses = 0
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}\n------------------")
        train(trainDataLoader, given_model, loss_fn, optimizer)
        accuracy, average_loss = test(testDataLoader, given_model, loss_fn)
        losses.append(average_loss)
        accuracies.append(accuracy)
        print(f"Epoch Average Loss: {average_loss}, Min Average loss: {average_loss_min}")
        print(f"Is Epoch Avg < Min Avg?: {average_loss < average_loss_min}")
        if average_loss < average_loss_min:
            average_loss_min = average_loss
            torch.save(given_model.state_dict(), "mid_epoch_box_model.pth")
            misses = 0
        elif epoch > minimum_epochs:
            print(f"Loss increased after {epoch+1} epochs")
            misses += 1
            if misses > forgiveness:
                print("miss!")
                given_model = ImageToHandBoxCNN().to(device)
                given_model.load_state_dict(torch.load("mid_epoch_box_model.pth", weights_only=False))
                break
    return losses, accuracies

In [8]:
losses, accuracies = train_epochs(30, model, loss_fn, optimizer)
torch.save(model.state_dict(), MODEL_PATH)

Epoch 1
------------------
Loss: 0.083136 [  128/ 2150]
Loss: 1.962304 [ 1408/ 2150]
Accuracy: 0.0
Average Loss: 50.79254302978516
Epoch Average Loss: 50.79254302978516, Min Average loss: inf
Is Epoch Avg < Min Avg?: True
Epoch 2
------------------
Loss: 0.395868 [  128/ 2150]
Loss: 0.086220 [ 1408/ 2150]
Accuracy: 0.0
Average Loss: 71.31315612792969
Epoch Average Loss: 71.31315612792969, Min Average loss: 50.79254302978516
Is Epoch Avg < Min Avg?: False
Epoch 3
------------------
Loss: 0.071165 [  128/ 2150]
Loss: 0.029166 [ 1408/ 2150]
Accuracy: 0.0
Average Loss: 22.6126220703125
Epoch Average Loss: 22.6126220703125, Min Average loss: 50.79254302978516
Is Epoch Avg < Min Avg?: True
Epoch 4
------------------
Loss: 0.026464 [  128/ 2150]
Loss: 0.020103 [ 1408/ 2150]
Accuracy: 6.826571809508473e-06
Average Loss: 4.079273843765259
Epoch Average Loss: 4.079273843765259, Min Average loss: 22.6126220703125
Is Epoch Avg < Min Avg?: True
Epoch 5
------------------
Loss: 0.014783 [  128/ 2150